# 05. Final Product Dashboard

Финальный product summary по сервису доставки еды: собираем в одном месте спрос, выручку, активность, повторные заказы и риск оттока.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import norm

orders = pd.read_csv(Path('../data/fact_orders.csv'), parse_dates=['order_ts'])
sessions = pd.read_csv(Path('../data/fact_app_sessions.csv'), parse_dates=['session_start_ts'])
ab = pd.read_csv(Path('../data/fact_ab_test_assignments.csv'), parse_dates=['assigned_at'])

orders = orders[orders['order_status'] == 'delivered'].copy()
orders['order_date'] = orders['order_ts'].dt.normalize()
orders['gmv_rub'] = orders['basket_value_rub'] + orders['delivery_fee_rub'] - orders['discount_rub']
sessions['session_date'] = sessions['session_start_ts'].dt.normalize()


In [2]:
orders_daily = orders.groupby('order_date').agg(
    orders=('order_id', 'count'),
    buyers=('user_id', 'nunique'),
    gmv=('gmv_rub', 'sum'),
    gross_margin=('gross_margin_rub', 'sum'),
    aov=('basket_value_rub', 'mean'),
).reset_index().rename(columns={'order_date': 'date'})

dau_daily = sessions.groupby('session_date')['user_id'].nunique().rename('dau').reset_index().rename(columns={'session_date': 'date'})
metrics = orders_daily.merge(dau_daily, on='date', how='left').fillna({'dau': 0})
metrics['buyer_conversion'] = metrics['buyers'] / metrics['dau'].replace(0, np.nan)
metrics['gmv_7d'] = metrics['gmv'].rolling(7, min_periods=1).mean()
metrics['orders_7d'] = metrics['orders'].rolling(7, min_periods=1).mean()
metrics['dau_7d'] = metrics['dau'].rolling(7, min_periods=1).mean()

first_orders = orders.groupby('user_id', as_index=False)['order_ts'].min().rename(columns={'order_ts': 'first_order_ts'})
repeat_orders = orders.merge(first_orders, on='user_id', how='inner')
repeat_orders = repeat_orders[repeat_orders['order_ts'] > repeat_orders['first_order_ts']].copy()
repeat_orders['days_after_first_order'] = (repeat_orders['order_ts'].dt.normalize() - repeat_orders['first_order_ts'].dt.normalize()).dt.days

max_order_ts = orders['order_ts'].max()
eligible_30 = first_orders[first_orders['first_order_ts'] <= max_order_ts - pd.Timedelta(days=30)].copy()
repeat_users_30 = repeat_orders.loc[
    repeat_orders['user_id'].isin(eligible_30['user_id'])
    & repeat_orders['days_after_first_order'].between(1, 30),
    'user_id'
].nunique()
repeat_rate_30d = repeat_users_30 / eligible_30['user_id'].nunique()

joined = ab.merge(orders[['user_id', 'order_ts']], on='user_id', how='left')
joined['within_7d'] = (joined['order_ts'] >= joined['assigned_at']) & (joined['order_ts'] < joined['assigned_at'] + pd.Timedelta(days=7))
user_conv = joined.groupby(['user_id', 'variant'], as_index=False)['within_7d'].max()
ab_summary = user_conv.groupby('variant').agg(users=('user_id', 'count'), converted=('within_7d', 'sum'))
ab_summary['conversion'] = ab_summary['converted'] / ab_summary['users']

p_c = float(ab_summary.loc['control', 'conversion'])
p_t = float(ab_summary.loc['treatment', 'conversion'])
n_c = int(ab_summary.loc['control', 'users'])
n_t = int(ab_summary.loc['treatment', 'users'])
p_pool = float(ab_summary['converted'].sum() / ab_summary['users'].sum())
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_c + 1 / n_t))
z = float((p_t - p_c) / se) if se else 0.0
p_value = float(2 * (1 - norm.cdf(abs(z))))
uplift_pp = (p_t - p_c) * 100

user_value = orders.groupby('user_id').agg(
    last_order_ts=('order_ts', 'max'),
    delivered_orders=('order_id', 'count'),
    gross_margin=('gross_margin_rub', 'sum'),
).reset_index()
user_value['days_inactive'] = (max_order_ts.normalize() - user_value['last_order_ts'].dt.normalize()).dt.days
churn_risk = user_value[user_value['days_inactive'] >= 28].copy()

kpi = pd.DataFrame({
    'Metric': [
        'Delivered orders',
        'GMV',
        'Gross margin',
        'Avg daily DAU',
        '30D repeat order rate',
        'A/B uplift',
        'Churn-risk users',
    ],
    'Value': [
        f"{len(orders):,}",
        f"{orders['gmv_rub'].sum():,.0f} RUB",
        f"{orders['gross_margin_rub'].sum():,.0f} RUB",
        f"{metrics['dau'].mean():,.0f}",
        f"{repeat_rate_30d:.1%}",
        f"{uplift_pp:+.2f} pp",
        f"{len(churn_risk):,}",
    ],
})
kpi.style.hide(axis='index')


Metric,Value
Delivered orders,"20,848"
GMV,"16,810,303 RUB"
Gross margin,"5,148,992 RUB"
Avg daily DAU,204
30D repeat order rate,57.3%
A/B uplift,-1.16 pp
Churn-risk users,"10,098"


In [3]:
fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}], [{'type': 'xy', 'secondary_y': True}, {'type': 'xy'}]],
    subplot_titles=('', '', 'Orders and GMV trend', 'DAU and buyer conversion'),
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

fig.add_trace(go.Indicator(
    mode='number',
    value=repeat_rate_30d * 100,
    number={'suffix': '%', 'valueformat': '.1f'},
    title={'text': '30D repeat order rate'},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode='number',
    value=round(uplift_pp, 2),
    number={'suffix': ' pp', 'valueformat': '+.2f'},
    title={'text': f'A/B uplift, p={p_value:.3f}'},
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=metrics['date'], y=metrics['orders_7d'], mode='lines', name='Orders, 7D avg', line=dict(color='#111827', width=3)
), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(
    x=metrics['date'], y=metrics['gmv_7d'], mode='lines', name='GMV, 7D avg', line=dict(color='#FFCC00', width=3)
), row=2, col=1, secondary_y=True)
fig.add_trace(go.Scatter(
    x=metrics['date'], y=metrics['dau_7d'], mode='lines', name='DAU, 7D avg', line=dict(color='#2563EB', width=3)
), row=2, col=2)
fig.add_trace(go.Scatter(
    x=metrics['date'], y=metrics['buyer_conversion'] * 100, mode='lines', name='Buyer conversion', line=dict(color='#10B981', width=3)
), row=2, col=2)

fig.update_layout(
    template='plotly_white',
    title='<b>Food Delivery Product Dashboard</b><br><sup>Core business health, repeat behavior, experiment signal, and retention risk</sup>',
    height=720,
    width=1150,
    margin=dict(l=70, r=45, t=105, b=70),
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Arial, sans-serif', size=13, color='#374151'),
)
fig.update_yaxes(title='Orders', row=2, col=1, secondary_y=False, gridcolor='#E5E7EB')
fig.update_yaxes(title='GMV, RUB', row=2, col=1, secondary_y=True, showgrid=False)
fig.update_yaxes(title='DAU / conversion, %', row=2, col=2, gridcolor='#E5E7EB')
fig.update_xaxes(showgrid=False)
fig.show()


In [4]:
signal_readout = (
    f"Treatment is higher by {uplift_pp:+.2f} pp; p-value={p_value:.3f}."
    if uplift_pp >= 0
    else f"Treatment is lower by {abs(uplift_pp):.2f} pp; p-value={p_value:.3f}."
)

summary_notes = pd.DataFrame({
    'Area': ['Retention', 'A/B test', 'Churn risk'],
    'Readout': [
        f"30D repeat order rate is {repeat_rate_30d:.1%}: more than half of eligible new users make a repeat order within 30 days.",
        signal_readout + ' The effect is not statistically confirmed at the 5% level.',
        f"{len(churn_risk):,} users are inactive for 28+ days; top-value users from this pool are candidates for CRM activation.",
    ],
})
summary_notes.style.hide(axis='index')


Area,Readout
Retention,30D repeat order rate is 57.3%: more than half of eligible new users make a repeat order within 30 days.
A/B test,Treatment is lower by 1.16 pp; p-value=0.376. The effect is not statistically confirmed at the 5% level.
Churn risk,"10,098 users are inactive for 28+ days; top-value users from this pool are candidates for CRM activation."
